# Pipelines CI/CD

L'intégration continue lance les tests à chaque mise à jour du dépôt `purchase_predict`.
Le déploiement continu enchaîne l'entraînement lorsque ces tests réussissent.
Ce notebook configure Cloud Build : secret de compte de service, `pytest`, puis `kedro run`.
Le déclencheur suit la branche `staging`.
L'adresse MLflow passe par une substitution, Cloud Build s'exécutant hors du VPC.


## Intégration, déploiement et environnements

L'intégration continue vérifie qu'une modification s'insère dans le dépôt par les tests et la fusion. Le déploiement continu publie ensuite cette version. Cloud Build enchaîne les étapes : une étape en échec arrête le build, donc l'entraînement ne part qu'après `pytest`.

La pré-production (`staging`) reproduit la production à plus petite échelle. Le service y est validé avec le reste de la plateforme avant la branche `master`. Cette séparation suit les interactions entre services : entraînement, registre de modèles, API.

Le flux d'entraînement comporte deux temps. Spark dépose les CSV dans Cloud Storage. Cloud Build lance Kedro, qui encode les données, optimise le modèle et l'enregistre dans MLflow. Plusieurs builds Kedro peuvent relire les mêmes CSV.


## Dépendances

`install.sh` prépare un environnement vierge. `requirements.txt` fige les versions, afin que le build installe les mêmes paquets qu'en production. Une étape Cloud Build n'a qu'un `entrypoint` : les commandes longues restent dans `install.sh`, enchaînées par `&&`.


## Secret d'authentification

L'image `<UTILISATEUR_GITHUB>/uv-gcp-cloud-build` exécute le Bash du build. La clé JSON du compte de service est fournie par Secret Manager, sous le nom attendu par le fichier : `cloud-build`, version `1`.

Dans Secret Manager, créer ce secret et y déposer la clé d'un compte autorisé sur Cloud Build, Compute Engine, Artifact Registry et Cloud Storage. Dans le fichier, `$$SERVICE_ACCOUNT` devient `$SERVICE_ACCOUNT` pour le shell. Un seul `$` serait consommé par les substitutions Cloud Build.


## Fichier cloudbuild.yaml

L'étape `CI` active le compte de service, installe les dépendances et lance `pytest`. L'étape `CD` relance l'installation puis `kedro run`. `ENV` reçoit `$BRANCH_NAME` (`staging` ou `master` selon le déclencheur) pour le versionnement MLflow. `MLFLOW_SERVER` reçoit la substitution `_MLFLOW_SERVER`.

Cloud Build est hors du VPC et ne résout pas le nom interne de MLflow. La substitution, ajoutée sur le déclencheur, porte l'adresse IP du serveur.

La machine par défaut et le délai de 10 minutes coupent un entraînement. Le fichier fixe donc la machine `N1_HIGHCPU_8` et un timeout de 1200 s. Pour un essai, le nombre d'itérations d'optimisation peut être ramené à 4. Les journaux partent dans le bucket `gs://purchase_predict`.


## Déclencheur

Créer le déclencheur `build-purchase-predict-staging` sur le dépôt `purchase_predict`, branche `staging`. Ajouter la substitution `_MLFLOW_SERVER` avec l'adresse IP de MLflow. Le push ci-dessous lance ce déclencheur. La clé SSH est chargée dans l'agent pour le remote `google`.
